[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/drdave-teaching/OPIM5509-notebooks/blob/main/Module1/1_CaliforniaHousing_EDA.ipynb)

# Module 1.1 - Exploratory Data Analysis (California Housing)

**OPIM 5509: Introduction to Deep Learning - University of Connecticut**

EDA is the part everyone wants to skip and nobody should. You are going to spend the semester feeding data into models that will happily learn nonsense if you hand them nonsense. This notebook is where we practice *looking first*.

The dataset is **California Housing** - 20,640 census block groups from the 1990 census, with the median home value in each one. It ships inside scikit-learn, so it loads in one line and it will still load in five years.

By the end you will have found two genuine landmines in this data that a careless modeler would drive straight over.

🔷 **The nugget:** EDA is not decoration. It is where you find the problems that would otherwise show up as a mysteriously good - or mysteriously bad - model score.

🔴
<!-- 🎙 DAVE TALKING POINTS (invisible when rendered - double-click to read):
Video 3 - EDA Part 1 - meeting the data and reading its shape

- OPEN with the framing: "we are refreshing, but I am going to teach EDA the way I want it done in your final project."
- Say why we left Boston Housing behind: scikit-learn REMOVED it in version 1.2 because one of its columns was an explicit racial proxy. Modern sklearn will not even fetch it. Be direct and brief - one minute, do not turn it into a lecture. Then move on.
- Load in one line. Emphasize: no Drive, no CSV, no upload. This is the standard for the whole course now.
- UNIT OF ANALYSIS is the big idea of this segment. Each row is a census BLOCK GROUP - roughly 600 to 3000 people - NOT a house. So "AveRooms" is average rooms per household in that block group. Students who forget this will write nonsense in their write-ups.
- Walk .shape, .columns, .dtypes, .info(), .isnull().sum(). Say the sentence: "info() gives me shape, dtypes, and missingness in one call."
- Point out this data has NO missing values and say why that is unrealistic - real data is messy, and Assignment 1 will be.
- Then .describe() and SLOW DOWN. This is where the two landmines are visible. Do not reveal them yet - ask the students to stare at the table and find the weird numbers. Tease into video 4.
-->

## 1. Load the data

One line. No file upload, no Drive mount, no path that breaks on someone else's machine.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
pd.set_option("display.width", 120)

from sklearn.datasets import fetch_california_housing

data = fetch_california_housing(as_frame=True)
df = data.frame          # features AND the target, together in one DataFrame
df.head()

## 2. What is a row?

This is the question to ask before any other question, and the one students most often skip.

**A row is a census block group, not a house.** A block group holds roughly 600 to 3,000 people. So every "Ave" column is an average *within that block group*, and `MedHouseVal` is the median home value *for that block group*.

| Column | What it actually means |
| --- | --- |
| `MedInc` | Median household income, in **tens of thousands** of dollars (`3.5` = \$35,000) |
| `HouseAge` | Median age of the houses, in years |
| `AveRooms` | Average rooms per household |
| `AveBedrms` | Average bedrooms per household |
| `Population` | People living in the block group |
| `AveOccup` | Average household members per household |
| `Latitude` / `Longitude` | Where the block group sits |
| `MedHouseVal` | **Target.** Median house value, in **hundreds of thousands** of dollars (`2.5` = \$250,000) |

**Remember:** if you cannot say in one sentence what a single row represents, you are not ready to model.

In [ ]:
# The three questions you ask of every new dataset
print("Shape (rows, columns):", df.shape)
print()
print("Column names:")
print(list(df.columns))

In [ ]:
# .info() answers shape, dtypes, and missingness all at once. Make this a reflex.
df.info()

In [ ]:
# Missing values, explicitly. Zero here - which is NOT what real data looks like.
print("Total missing values in the whole table:", df.isnull().sum().sum())
print()
print(df.isnull().sum())

**Caution:** zero missing values is a sign you are working with a *teaching* dataset. Assignment 1 hands you data with missing values in every column, because that is the normal case. Do not build the habit of skipping the missingness check just because it was clean this once.

## 3. `describe()` - and the two things hiding in it

Run the cell below and actually *read* it. Look at the `max` row. At least two of these columns contain values that cannot possibly be literally true, and one column has a ceiling that will quietly cap your model's accuracy.

Find them before you scroll.

In [ ]:
df.describe().T

🔴
<!-- 🎙 DAVE TALKING POINTS (invisible when rendered - double-click to read):
Video 4 - EDA Part 2 - finding the landmines, then visualizing

- PAY OFF the cliffhanger. Two landmines:
  (1) MedHouseVal is CAPPED at 5.00001 (i.e. $500,001). About 965 block groups sit exactly on that ceiling.
      Show the value_counts() cell. Show the histogram spike on the right edge. This is censored data.
      Connect it back to Boston Housing where medv topped out at exactly 50 - same phenomenon, students
      who took my course before will recognize it.
      Consequence: no model can ever predict above 5, and every expensive neighborhood is squashed together.
  (2) AveRooms max is ~141.9 and AveOccup max is ~1243. A household does not have 141 rooms or 1243 people.
      These are block groups with a tiny number of households - a divide-by-almost-nothing artifact.
      Show the sorted tail. Show that they are a handful of rows.
- The judgment call: do we delete them? Say honestly "it depends" and give the rule - understand the
  mechanism first, then decide. Do NOT let them think deleting outliers is automatic.
- Then move to plots. Order matters: univariate (histograms, boxplot) -> bivariate (scatter, correlation)
  -> geographic. Build up.
- The correlation heatmap: MedInc is the strongest single predictor at ~0.69. Say the sentence:
  "income predicts home value - that is not surprising, and a model that DID NOT find this would be broken."
- AveRooms and AveBedrms are ~0.85 correlated. Name it: multicollinearity. Tell them trees do not care much,
  linear regression does. This pays off in notebook 2.
- The lat/lon scatter is the money shot. Coast is expensive, Central Valley is cheap, and you can SEE
  San Francisco and Los Angeles appear out of nothing but two numeric columns. Let it land.
- CLOSE by tying to deep learning: everything we just did by eye, a network does by weight. But the network
  cannot tell you the target was censored. Only you can.
-->

## 4. Landmine one: the target is capped

Look at the maximum of `MedHouseVal`: `5.00001`. That is not a coincidence, and it is not a rounding artifact. The 1990 census **top-coded** home values at \$500,001 - anything more expensive was recorded as exactly that.

This is **censored data**, and it has a direct consequence for every model you will build this semester: no model can predict above the cap, and every genuinely expensive neighborhood has been squashed into a single value.

In [ ]:
# How many block groups sit exactly on the ceiling?
capped = (df["MedHouseVal"] >= 5.0).sum()
print(f"Block groups at the cap: {capped}  ({capped / len(df):.1%} of the data)")
print()
print("The five most common target values:")
print(df["MedHouseVal"].value_counts().head())

In [ ]:
# You can SEE the cap. Look at the spike on the right edge.
plt.figure(figsize=(9, 4))
plt.hist(df["MedHouseVal"], bins=60, color="#C0202C", edgecolor="white")
plt.axvline(5.0, color="black", linestyle="--", linewidth=1.5)
plt.text(4.95, plt.ylim()[1] * 0.9, " the cap ", ha="right", fontsize=10)
plt.xlabel("Median house value ($100,000s)")
plt.ylabel("Number of block groups")
plt.title("California Housing: the target is censored at $500,001")
plt.tight_layout()
plt.show()

🔷 **The nugget:** a histogram of your target takes four seconds and tells you whether your problem is even well-posed. Do it every single time - and if you took OPIM 5512 with me, you saw this exact spike in Boston Housing where `medv` topped out at 50.

## 5. Landmine two: impossible averages

`AveRooms` has a maximum around 141.9 and `AveOccup` reaches roughly 1,243. No household has 141 rooms, and none has 1,243 people.

These are real records, not typos. They come from block groups with very few households - the average is a small number divided by an even smaller one. That is a **mechanism**, and understanding the mechanism is what lets you decide what to do.

In [ ]:
# Who are these rows? Sort and look at the tail.
odd = df.sort_values("AveOccup", ascending=False).head(6)
odd[["AveRooms", "AveBedrms", "Population", "AveOccup", "MedHouseVal"]]

In [ ]:
# How many rows are genuinely extreme? Very few - which is itself useful to know.
extreme = df[(df["AveOccup"] > 10) | (df["AveRooms"] > 20)]
print(f"Extreme rows: {len(extreme)} out of {len(df)}  ({len(extreme) / len(df):.2%})")

**Caution:** "outlier" is not a synonym for "delete." The right question is *why does this value exist?* Here the answer is a small-denominator artifact, so dropping a handful of rows is defensible. If instead these had been your most valuable customers, deleting them would have destroyed the whole analysis.

Understand the mechanism, then decide. Never the other way around.

## 6. Univariate: what does each column look like on its own?

In [ ]:
df.hist(figsize=(12, 8), bins=40, color="#C0202C", edgecolor="white")
plt.suptitle("Distribution of every column", y=1.0, fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# A boxplot makes the skew and the outliers unmistakable.
fig, axes = plt.subplots(1, 3, figsize=(12, 4))
for ax, col in zip(axes, ["MedInc", "HouseAge", "MedHouseVal"]):
    ax.boxplot(df[col], vert=True, widths=0.5)
    ax.set_title(col)
    ax.set_xticks([])
plt.suptitle("Boxplots: the box is the middle 50%, the dots are the tails")
plt.tight_layout()
plt.show()

## 7. Feature engineering: a flag variable

A flag (dummy, indicator, binary) variable recodes something continuous into a 0 or a 1. It is the simplest feature you can build, it makes group comparisons easy, and in notebook 3 the exact same idea turns this regression problem into a classification problem.

In [ ]:
# 1 if this block group is above the statewide median value, else 0
df["FLAG_expensive"] = np.where(df["MedHouseVal"] > df["MedHouseVal"].median(), 1, 0)

print("Median target value:", round(df["MedHouseVal"].median(), 3))
print()
print(df["FLAG_expensive"].value_counts())

In [ ]:
# Now a group-by (a pivot table) - how do expensive and cheap block groups differ?
df.groupby("FLAG_expensive")[["MedInc", "HouseAge", "AveRooms", "Population", "AveOccup"]].mean().round(2)

Read that table out loud: expensive block groups have substantially higher median income and slightly more rooms per household. Population is nearly identical. That is a finding, and it is the kind of sentence that belongs in your write-up.

## 8. Bivariate: how do the columns relate?

In [ ]:
plt.figure(figsize=(9, 7))
sns.heatmap(df.drop(columns="FLAG_expensive").corr(numeric_only=True),
            annot=True, fmt=".2f", cmap="RdBu_r", center=0, square=True,
            cbar_kws={"shrink": 0.8})
plt.title("Correlation matrix")
plt.tight_layout()
plt.show()

Two things to take from that heatmap:

1. **`MedInc` is the strongest single predictor of `MedHouseVal`** (about 0.69). Income predicts home value. That is not a surprise - and a model that *failed* to find it would be broken.
2. **`AveRooms` and `AveBedrms` are correlated around 0.85.** That is **multicollinearity**: two columns carrying nearly the same information. Tree-based models shrug it off. Linear regression does not - coefficients get unstable and hard to interpret. Watch for this in notebook 2.

In [ ]:
# The single most important scatterplot in this dataset
plt.figure(figsize=(8, 5))
plt.scatter(df["MedInc"], df["MedHouseVal"], alpha=0.15, s=6, color="#C0202C")
plt.axhline(5.0, color="black", linestyle="--", linewidth=1)
plt.xlabel("Median income ($10,000s)")
plt.ylabel("Median house value ($100,000s)")
plt.title("Income vs. house value - notice the flat line of capped values along the top")
plt.tight_layout()
plt.show()

That dashed line at the top is the censoring from section 4, now visible as a *stripe* of points. The relationship is real and strong, but it flattens out at the ceiling.

## 9. Geography as EDA

`Latitude` and `Longitude` are just two more numeric columns. Plot one against the other, color by price, and California draws itself - coastline, Bay Area, Los Angeles, and a cheap Central Valley down the middle.

In [ ]:
plt.figure(figsize=(8, 8))
sc = plt.scatter(df["Longitude"], df["Latitude"],
                 c=df["MedHouseVal"], cmap="viridis",
                 alpha=0.45, s=6)
plt.colorbar(sc, label="Median house value ($100,000s)", shrink=0.75)
plt.xlabel("Longitude")
plt.ylabel("Latitude")
plt.title("California house values - the map appears from two numeric columns")
plt.tight_layout()
plt.show()

🔷 **The nugget:** you did not load a shapefile. Two ordinary numeric columns and a scatterplot reproduced the state of California and its housing market. Always ask what your columns *mean* before you decide they are boring.

## 10. Cleaning, now that we understand why

We drop the small-denominator artifacts. We **keep** the capped target rows - they are real observations, and throwing away every expensive neighborhood would bias the model badly. Instead we stay honest about the ceiling when we report results.

In [ ]:
before = len(df)

df_clean = df[(df["AveOccup"] <= 10) & (df["AveRooms"] <= 20)].copy()

print(f"Rows before: {before}")
print(f"Rows after : {len(df_clean)}   (dropped {before - len(df_clean)})")
print()
df_clean[["AveRooms", "AveOccup"]].describe().T

## What you should have after this notebook

- You can state what a single row represents before touching a model
- `.info()`, `.describe()`, and `.isnull().sum()` are reflexes, not steps you look up
- You found a **censored target** and can explain what it does to model performance
- You found **impossible averages**, explained the mechanism, and made a defensible call
- You can read a correlation heatmap and name **multicollinearity** when you see it
- You built a flag variable and used a group-by to compare two populations
- You know that latitude and longitude are features, not metadata

**On your own:** re-run section 8's scatterplot with `AveOccup` on the x-axis instead of `MedInc`. Is the relationship as strong? Why not?

---

**Next:** `2_AllTheModels_Regression.ipynb` - the same data, now with models on top.